# Probing por cabeza sobre atributos de AwA2, con `reward_similarity` (2026-10-17)

Mismo experimento y mismos ajustes que `notebooks/25092026_awa2_probing/probing_awa2.ipynb` (bloque 4, token CLS, 47 atributos visuales, probe lineal Ridge con R² fuera de muestra, validación cruzada por imagen y por clase), reutilizando su código tal cual, pero **añadiendo el modelo `reward_similarity`** (`lambda_loss = -0.1`: se premia la similitud entre cabezas) a los tres que ya se comparaban. Así los números son comparables con los del notebook original.

Se incluye el probe por cabeza, el probe sobre las 4 cabezas concatenadas, y un resumen de qué tan parecidas son las cabezas por lo que decodifican. No se repiten el null de permutación ni la selectividad del notebook original.

In [1]:
# ======================================================================
MODELS = {"unconstrained": "vit-15-AwA2", "constrained": "vit-15-constrained-AwA2", "constrained_rank": "vit-15-constrained_rank-AwA2",
          "reward_similarity": "vit-15-reward_similarity-AwA2"}
BLOCK_IDX = 4             # 1-indexed, como en ablation.ipynb / head_specialization.ipynb
TOKEN = "cls"             # "cls" | "mean_patches" (promedio de la salida de la cabeza sobre los 64 parches)
N_FOLDS = 5
R2_THRESHOLD = 0.3        # "atributo bien predicho" por una cabeza (se reporta también un barrido)
BALACC_THRESHOLD = 0.75
SEED = 0

# Filtro de ruido: solo atributos con correlato visual directo en una imagen estática
# (taxonomía de categorías de Lampert et al., AwA2). Se excluyen comportamiento/locomoción,
# actividad, dieta, estrategia de forrajeo, carácter/temperamento y geografía (newworld/oldworld),
# que no son algo que un probe sobre una imagen pueda ver, solo inferir por correlación con la especie.
VISUAL_ATTRS = {
    # color (8)
    "black", "white", "blue", "brown", "gray", "orange", "red", "yellow",
    # textura (6)
    "patches", "spots", "stripes", "furry", "hairless", "toughskin",
    # forma (4)
    "big", "small", "bulbous", "lean",
    # partes del cuerpo (15)
    "flippers", "hands", "hooves", "pads", "paws", "longleg", "longneck", "tail",
    "chewteeth", "meatteeth", "buckteeth", "strainteeth", "horns", "claws", "tusks",
    # hábitat / escenario visible en la foto (14, sin newworld/oldworld)
    "arctic", "coastal", "desert", "bush", "plains", "forest", "fields", "jungle",
    "mountains", "ocean", "ground", "water", "tree", "cave",
}
# ======================================================================

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import balanced_accuracy_score, r2_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root))

from modules.datasets import AwA2Dataset
from modules.model import ViTForClassfication

torch.manual_seed(SEED)
device = torch.device("cpu")

EXPERIMENTS_DIR = project_root / "experimentation" / "experiments"
OUTPUT_DIR = EXPERIMENTS_DIR / "17102026_probing_reward"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = f"block{BLOCK_IDX}_{TOKEN}_visual"


def save_fig(fig, stem):
    fig.savefig(OUTPUT_DIR / f"{stem}_{RUN_TAG}.png", dpi=150, bbox_inches="tight")

In [2]:
dataset = AwA2Dataset(image_size=32, root=str(project_root / "data" / "awa2"))
_, testloader, classes = dataset.get_loaders(batch_size=256)
attr_names = list(dataset.attribute_names)

labels = dataset.testset.labels.numpy()
Y_cont = dataset.attributes_continuous[labels].numpy()   # (N, 85) en [0, 1]
Y_bin = dataset.attributes_binary[labels].numpy().astype(int)

# Filtro de ruido no-visual (ver VISUAL_ATTRS en la config): de acá en adelante todo el notebook
# trabaja solo sobre atributos con correlato visual directo.
visual_mask = np.array([name in VISUAL_ATTRS for name in attr_names])
dropped = [name for name in attr_names if name not in VISUAL_ATTRS]
attr_names = [name for name in attr_names if name in VISUAL_ATTRS]
Y_cont = Y_cont[:, visual_mask]
Y_bin = Y_bin[:, visual_mask]
n_attr = len(attr_names)
print(f"atributos: {n_attr} visuales de {len(visual_mask)} totales ({len(dropped)} descartados)")
print(f"test: {len(labels)} imágenes, {len(classes)} clases, {n_attr} atributos")


def load_model(name):
    exp_dir = EXPERIMENTS_DIR / name
    config = json.loads((exp_dir / "config.json").read_text())
    metrics = json.loads((exp_dir / "metrics.json").read_text())
    assert not config["use_faster_attention"], "se asume block.attention.heads como submódulos separados"
    model = ViTForClassfication(config)
    model.load_state_dict(torch.load(exp_dir / "model_final.pt", map_location="cpu"))
    return model.to(device).eval(), config, metrics


models = {key: load_model(name) for key, name in MODELS.items()}
n_heads = models["unconstrained"][1]["num_attention_heads"]
block_idx0 = BLOCK_IDX - 1
head_labels = [f"head {h + 1}" for h in range(n_heads)]

atributos: 47 visuales de 85 totales (38 descartados)
test: 7468 imágenes, 50 clases, 47 atributos


In [3]:
@torch.no_grad()
def extract_head_outputs(model):
    captured = {}
    heads = model.encoder.blocks[block_idx0].attention.heads
    hooks = [head.register_forward_hook(lambda _m, _i, out, h=h: captured.__setitem__(h, out[0]))
             for h, head in enumerate(heads)]
    feats, preds = [], []
    try:
        for images, _, _ in testloader:
            logits, _ = model(images.to(device))
            per_head = torch.stack([captured[h] for h in range(n_heads)], dim=1)   # (B, H, T, d)
            per_head = per_head[:, :, 0] if TOKEN == "cls" else per_head[:, :, 1:].mean(dim=2)
            feats.append(per_head.cpu())
            preds.append(logits.argmax(-1).cpu())
    finally:
        for hook in hooks:
            hook.remove()
    return torch.cat(feats).numpy(), torch.cat(preds).numpy()


head_feats = {}
for key, (model, _, metrics) in models.items():
    head_feats[key], preds = extract_head_outputs(model)
    acc = (preds == labels).mean()
    print(f"{key:>13}: feats {head_feats[key].shape}, accuracy recomputada {acc:.4f} "
          f"(metrics.json: {metrics['accuracy']:.4f})")

unconstrained: feats (7468, 4, 12), accuracy recomputada 0.2971 (metrics.json: 0.2971)


  constrained: feats (7468, 4, 12), accuracy recomputada 0.2879 (metrics.json: 0.2879)


constrained_rank: feats (7468, 4, 12), accuracy recomputada 0.2973 (metrics.json: 0.2973)


reward_similarity: feats (7468, 4, 12), accuracy recomputada 0.2631 (metrics.json: 0.2631)


In [4]:
folds_image = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(labels, labels))
folds_class = list(GroupKFold(N_FOLDS).split(labels, labels, groups=labels))   # clases no vistas


def probe_r2(X, Y, folds):
    """R² out-of-fold por atributo (Ridge multi-salida, alpha elegido por atributo dentro de cada fold)."""
    pred = np.zeros_like(Y)
    for tr, te in folds:
        probe = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13), alpha_per_target=True))
        pred[te] = probe.fit(X[tr], Y[tr]).predict(X[te])
    return r2_score(Y, pred, multioutput="raw_values")


def probe_balacc(X, Yb, folds):
    """Balanced accuracy out-of-fold por atributo binarizado (logística con class_weight balanceado)."""
    scores = np.full(Yb.shape[1], np.nan)
    for a in range(Yb.shape[1]):
        y = Yb[:, a]
        if y.min() == y.max():
            continue
        pred = np.zeros_like(y)
        for tr, te in folds:
            probe = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000))
            pred[te] = probe.fit(X[tr], y[tr]).predict(X[te])
        scores[a] = balanced_accuracy_score(y, pred)
    return scores


r2, r2_unseen, balacc = {}, {}, {}
r2_block, r2_block_unseen, balacc_block = {}, {}, {}
for key, feats in head_feats.items():
    r2[key] = np.stack([probe_r2(feats[:, h], Y_cont, folds_image) for h in range(n_heads)])          # (H, 85)
    r2_unseen[key] = np.stack([probe_r2(feats[:, h], Y_cont, folds_class) for h in range(n_heads)])
    balacc[key] = np.stack([probe_balacc(feats[:, h], Y_bin, folds_image) for h in range(n_heads)])
    # Probe sobre las 4 cabezas concatenadas (48-d), por atributo: lo decodificable del bloque cuando las
    # cabezas se usan juntas. Mismas métricas y mismos folds que el probe por cabeza.
    X_block = feats.reshape(len(feats), -1)
    r2_block[key] = probe_r2(X_block, Y_cont, folds_image)                       # (n_attr,)
    r2_block_unseen[key] = probe_r2(X_block, Y_cont, folds_class)
    balacc_block[key] = probe_balacc(X_block, Y_bin, folds_image)
    print(f"{key:>13}: R² medio {r2[key].mean():.3f} | clases no vistas {r2_unseen[key].mean():.3f} | "
          f"bal.acc media {np.nanmean(balacc[key]):.3f}")
    print(f"{'':>13}  4 cabezas (48-d): R² {r2_block[key].mean():.3f} | clases no vistas "
          f"{r2_block_unseen[key].mean():.3f} | bal.acc {np.nanmean(balacc_block[key]):.3f}")

for key in MODELS:
    pd.DataFrame(r2[key], index=head_labels, columns=attr_names).to_csv(OUTPUT_DIR / f"r2_{key}_{RUN_TAG}.csv")
    pd.DataFrame(r2_unseen[key], index=head_labels, columns=attr_names).to_csv(OUTPUT_DIR / f"r2_unseen_classes_{key}_{RUN_TAG}.csv")
    pd.DataFrame(balacc[key], index=head_labels, columns=attr_names).to_csv(OUTPUT_DIR / f"balacc_{key}_{RUN_TAG}.csv")
    pd.DataFrame({"R2": r2_block[key], "R2_unseen_classes": r2_block_unseen[key], "balacc": balacc_block[key]},
                 index=attr_names).to_csv(OUTPUT_DIR / f"block_concat_{key}_{RUN_TAG}.csv")

unconstrained: R² medio 0.076 | clases no vistas -0.022 | bal.acc media 0.630
               4 cabezas (48-d): R² 0.161 | clases no vistas 0.007 | bal.acc 0.695


  constrained: R² medio 0.044 | clases no vistas -0.048 | bal.acc media 0.607
               4 cabezas (48-d): R² 0.123 | clases no vistas -0.035 | bal.acc 0.677


constrained_rank: R² medio 0.047 | clases no vistas -0.039 | bal.acc media 0.604
               4 cabezas (48-d): R² 0.116 | clases no vistas -0.022 | bal.acc 0.664


reward_similarity: R² medio 0.092 | clases no vistas -0.012 | bal.acc media 0.640
               4 cabezas (48-d): R² 0.166 | clases no vistas -0.004 | bal.acc 0.687


## Resumen comparativo

- **R² medio / clases no vistas:** cuánta información de atributos hay en una cabeza (por imagen y generalizando a clases no vistas al ajustar el probe).
- **Ganancia concat − mejor cabeza:** cuánto se gana al leer las 4 cabezas juntas frente a la mejor por separado.
- **Coseno entre cabezas (vectores de R² por atributo):** si dos cabezas decodifican los mismos atributos (alto) o distintos (bajo). Es la lectura semántica de la redundancia entre cabezas.

In [5]:
def offdiag_cosine(R):
    Rp = np.clip(R, 0, None)
    Rn = Rp / np.clip(np.linalg.norm(Rp, axis=1, keepdims=True), 1e-12, None)
    M = Rn @ Rn.T
    return float(M[np.triu_indices(len(M), k=1)].mean())

rows = {}
for key in MODELS:
    rows[key] = {
        "accuracy (test)": models[key][2]["accuracy"],
        "R² medio (cabeza × atributo)": float(r2[key].mean()),
        "R² medio, clases no vistas": float(r2_unseen[key].mean()),
        "bal.acc media": float(np.nanmean(balacc[key])),
        "R² 4 cabezas (48-d)": float(r2_block[key].mean()),
        "R² 4 cabezas, clases no vistas": float(r2_block_unseen[key].mean()),
        "ganancia concat − mejor cabeza (R²)": float((r2_block[key] - r2[key].max(0)).mean()),
        "coseno medio entre cabezas (vectores de R²)": offdiag_cosine(r2[key]),
        **{f"R² medio, {head_labels[h]}": float(r2[key][h].mean()) for h in range(n_heads)},
    }
summary = pd.DataFrame(rows)
print(summary.round(3).to_string())
summary.to_csv(OUTPUT_DIR / f"summary_with_reward_{RUN_TAG}.csv")

print("\nTop 5 atributos por R² (4 cabezas concatenadas):")
for key in MODELS:
    top = pd.Series(r2_block[key], index=attr_names).sort_values(ascending=False).head(5)
    print(f"{key:>18}: " + ", ".join(f"{a} ({v:.2f})" for a, v in top.items()))

                                             unconstrained  constrained  constrained_rank  reward_similarity
accuracy (test)                                      0.297        0.288             0.297              0.263
R² medio (cabeza × atributo)                         0.076        0.044             0.047              0.092
R² medio, clases no vistas                          -0.022       -0.048            -0.039             -0.012
bal.acc media                                        0.630        0.607             0.604              0.640
R² 4 cabezas (48-d)                                  0.161        0.123             0.116              0.166
R² 4 cabezas, clases no vistas                       0.007       -0.035            -0.022             -0.004
ganancia concat − mejor cabeza (R²)                  0.060        0.051             0.049              0.066
coseno medio entre cabezas (vectores de R²)          0.961        0.905             0.945              0.995
R² medio, head 1   